In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device ='cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size=8
batch_size=4

cuda


Step 1: Read input data
---
Step 2: Find all characters used in that 
---


In [3]:
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text=f.read()
chars = sorted(set(text))
print(chars)
vocabulary_size=len(chars)
print(vocabulary_size)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
89


Step 3: Based on all characters make a vocabulary list/Dictionary which you will use for tokeniser
and assign a number to each token
---
Step 4: Make encoder and Decoder- i.e. given a text iput you can convert it to number using encoder function and back to text using decoder function
---

In [4]:
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode= lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])
# I used character level tokeniser

Below is example for a word coverted using encoder and converted back using decoder

In [5]:
encoded_hello= encode('hello')
print(encoded_hello)
decoded_hello= decode(encoded_hello)
print(decoded_hello)

[63, 60, 67, 67, 70]
hello


Step 5: Convert text input to Tokens and store them in tensor datatype
---

In [6]:
data= torch.tensor(encode(text), dtype=torch.long )
print(data[:100])

tensor([ 0, 33, 70, 73, 70, 75, 63, 80,  1, 56, 69, 59,  1, 75, 63, 60,  1, 52,
        64, 81, 56, 73, 59,  1, 64, 69,  1, 44, 81,  0,  0,  0,  1,  1, 30,  1,
        35, 56, 64, 75, 63, 61, 76, 67,  1, 47, 60, 58, 70, 73, 59,  1, 70, 61,
         1, 49, 63, 60, 64, 73,  1, 30, 68, 56, 81, 64, 69, 62,  1, 30, 59, 77,
        60, 69, 75, 76, 73, 60, 74,  0,  1,  1,  1,  1, 64, 69,  1, 56, 69,  1,
        50, 69, 59, 60, 73, 62, 73, 70, 76, 69])


Step 6: Now Start by defing train,val set and block size which we will use for training
---

In [7]:
n = int(0.8*len(data))
train_data= data[:n]
val_data= data[n:]

def get_batch(split):
    data=train_data if(split)=="train" else val_data
    ix= torch.randint(len(data)-block_size,(batch_size,))
    print(ix)
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+1+block_size] for i in ix])
    x,y=x.to(device), y.to(device)
    return x,y


x,y=get_batch('train')
print('inputs',x)
print('target',y)


tensor([117710,  84381,  51521,  79722])
inputs tensor([[ 1, 75, 70, 71, 13,  1, 69, 70],
        [70, 69, 75, 60, 69, 75, 60, 59],
        [73,  1, 71, 60, 70, 71, 67, 60],
        [78, 60, 59,  1, 56, 75,  1, 63]], device='cuda:0')
target tensor([[75, 70, 71, 13,  1, 69, 70, 78],
        [69, 75, 60, 69, 75, 60, 59,  1],
        [ 1, 71, 60, 70, 71, 67, 60, 15],
        [60, 59,  1, 56, 75,  1, 63, 64]], device='cuda:0')


In [8]:


x= train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context= x[0:t+1]
    target= y[t]
    print('when input is', context, 'target is', target)


when input is tensor([0]) target is tensor(33)
when input is tensor([ 0, 33]) target is tensor(70)
when input is tensor([ 0, 33, 70]) target is tensor(73)
when input is tensor([ 0, 33, 70, 73]) target is tensor(70)
when input is tensor([ 0, 33, 70, 73, 70]) target is tensor(75)
when input is tensor([ 0, 33, 70, 73, 70, 75]) target is tensor(63)
when input is tensor([ 0, 33, 70, 73, 70, 75, 63]) target is tensor(80)
when input is tensor([ 0, 33, 70, 73, 70, 75, 63, 80]) target is tensor(1)


In [26]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table= nn.Embedding(vocabulary_size,vocabulary_size)
        

    def forward(self, index, targets=None):
        logits=self.token_embedding_table(index)
        #print(logits.shape)

        if targets is None:
            loss=None
        else:
            B, T, C= logits.shape
            print(B,T,C)
            logits= logits.view(B*T,C)
            targets= targets.view(B*T)
            loss= F.cross_entropy(logits, targets)
        return logits,loss
    def generate(self, index, maxnewtokens):
        # index is (B,T) array of indices in the current context
        for _ in range(maxnewtokens):
            # get the predictions
            logits,loss= self.forward(index)
            # focus only on last time step
            logits= logits[:,-1,:] # becomes (B,C)
            # Apply softmax to get probablities
            probs= F.softmax(logits, dim=-1) #(B,C)
            # sample from distribution
            index_next=torch.multinomial(probs,num_samples=1)
            # append sampled index to the running sequence
            index=torch.cat((index,index_next),dim=1)
            
        return index

model=BigramLanguageModel(vocabulary_size)
m=model.to(device)

context=torch.zeros((1,1),dtype=torch.long, device=device)
generated_chars=decode(m.generate(context,maxnewtokens=500)[0].tolist())
print(generated_chars)
print(context)




S3‘Q3*4O‘Ll9 iYSLM$X1kVY,q$#:'(ydTQ$kx.1$; YXWvWjZ6)d-•c4’NZ3eTUTvs’)C&QmP’bV0$;™*$k/”cx-Lt%!•uh
QnNGZS%—YEOs,i6lnO3jt:X™d"W:'p)!u&DMXE%H”mD.tj!y•J‘"7D3rv !4E'&t/b/RKH*F(q&"’r2—/K•v6YvcS-B8!:Hi3‘J6m%!bv;vX8vp’npCm$xgfKAcH7(v‘,%‘L;i+lRq#IHg8cADYw—C*h2k/—?”:SB5VDw—#ZWv—ju2&nq-3NFjlYxfkQn ?$wNFS)Gs*)Cc*GL
1ae,LW:$o%!RK2W4 D*•vlD73ZIA4jfCzrj7f’Gm$-lud-B8—dimE4;;o:—"Zlln.”kJHR4)"Tbg-l0zofV”1D*/NGD*';22+F52“)ONE#Z.(Tj(UxZp5d
u.kh™T&QlB5z)*;™POgT’wuIg"v*GL9 -VZI7•*•E‘789NobYR+f:gdW7c7z((IJ#)&(:g:3Js7F”
tensor([[0]], device='cuda:0')
